# Experiment 2: Early Exit INT8 ResNet

This experiment adds an auxiliary "early exit" classifier midway through the network (after `layer2`).

If the early classifier is highly confident (Entropy < Threshold), we exit early and save computation! We will use a fixed INT8 backbone (simulated on GPU) for an apples-to-apples comparison.

In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git 2>/dev/null; true
import sys
sys.path.append('Quantization_project')

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time, numpy as np, matplotlib.pyplot as plt

from experiment2 import early_exit_int8_resnet20

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

## Training Setup

In [ ]:
model = early_exit_int8_resnet20().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

EPOCHS = 200
ALPHA = 0.3  # Weight of the early exit loss
INT8_MULTIPLIER = 8 * 8  # FLOPs multiplier for INT8

In [ ]:
train_losses, test_accs = [], []

try:
    for epoch in range(EPOCHS):
        model.train()
        running_loss, correct, total = 0, 0, 0

        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            early_logits, final_logits = model(inputs)
            
            loss_final = criterion(final_logits, targets)
            loss_early = criterion(early_logits, targets)
            loss = loss_final + ALPHA * loss_early
            
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = final_logits.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        scheduler.step()
        train_acc = 100. * correct / total
        train_losses.append(running_loss / len(trainloader))

        # Simple Evaluation (Always run full network for test tracking)
        model.eval()
        t_correct, t_total = 0, 0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs, _ = model(inputs, entropy_threshold=-1.0) # threshold < 0 ensures NO early exits
                _, predicted = outputs.max(1)
                t_total += targets.size(0)
                t_correct += predicted.eq(targets).sum().item()
                
        test_acc = 100. * t_correct / t_total
        test_accs.append(test_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_losses[-1]:.3f} | Train Acc: {train_acc:.2f}% | Test Acc (Full Network): {test_acc:.2f}%')

except KeyboardInterrupt:
    print("\nTraining interrupted early!")

torch.save(model.state_dict(), 'early_exit_resnet20.pth')
print('Model saved.')

## Threshold Sweep & Evaluation
We evaluate the model using various entropy thresholds. A higher threshold means the model is more willing to exit early (higher confidence requirement to continue).

In [ ]:
thresholds = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
results = []

model.eval()
for thresh in thresholds:
    correct, total, early_exits = 0, 0, 0
    
    # Measure Accuracy and FLOPs
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs, confident_mask = model(inputs, entropy_threshold=thresh)
            
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            early_exits += confident_mask.sum().item()
            
    acc = 100. * correct / total
    exit_rate = early_exits / total
    
    # Calculate FLOPs
    base_flops_total = model.flops_base * total
    layer3_flops_total = model.flops_layer3 * (total - early_exits)
    eff_flops = (base_flops_total + layer3_flops_total) * INT8_MULTIPLIER
    eff_flops_per_img = eff_flops / total
    
    # Measure Latency (30 runs)
    times = []
    with torch.no_grad():
        for _ in range(30):
            if torch.cuda.is_available(): torch.cuda.synchronize()
            start = time.time()
            for inputs, _ in testloader:
                inputs = inputs.to(device)
                _ = model(inputs, entropy_threshold=thresh)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            times.append(time.time() - start)
    lat = np.mean(times)
    
    results.append({
        'threshold': thresh,
        'acc': acc,
        'exit_rate': exit_rate * 100,
        'latency': lat,
        'flops': eff_flops_per_img
    })
    
    print(f'Thresh: {thresh:3.1f} | Acc: {acc:5.2f}% | Early Exits: {exit_rate*100:5.1f}% | Latency: {lat:.4f}s | FLOPs: {eff_flops_per_img/1e6:5.1f}M')

In [ ]:
accs = [r['acc'] for r in results]
lats = [r['latency'] for r in results]
rates = [r['exit_rate'] for r in results]
flops = [r['flops']/1e6 for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lats, accs, 'o-', color='#FF5722', linewidth=2, markersize=8)
axes[0].set_xlabel('Inference Latency (s)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy vs Latency Trade-off')
axes[0].grid(True, alpha=0.3)
for i, r in enumerate(results):
    axes[0].annotate(f"{r['threshold']}", (lats[i], accs[i]), textcoords="offset points", xytext=(0,5), ha='center')

axes[1].plot(rates, accs, 'o-', color='#4CAF50', linewidth=2, markersize=8)
axes[1].set_xlabel('Early Exit Rate (%)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy vs Early Exit Rate')
axes[1].grid(True, alpha=0.3)
axes[1].invert_xaxis() # Lower exit rate usually means higher accuracy

plt.tight_layout()
plt.savefig('early_exit_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()